# Grant App Permissions

Grants required permissions on the Databricks App after deploy:
1. **M2M SPN → CAN_USE on App** — required so the MCP HTTP connection can authenticate through the app's auth sidecar
2. **M2M SPN → UC grants** — USE CATALOG, USE SCHEMA, EXECUTE on MCP service (`account users` does NOT include SPNs)
3. **M2M SPN → READ VOLUME + WRITE VOLUME on `ui_visual_checks`** (optional `visual_checks_volume` parameter; build 5 wave 5-5). The volume is created by ground-truth-infra, which deploys first, so it always exists here.

App-level access uses the workspace Permissions API; UC grants use the Grants API. Both are Python (not SQL DDL).

Run by `deploy.sh` Phase 2 via `databricks bundle run app_post_deploy_setup`.
Also safe to run manually — `update_permissions` is idempotent (PATCH).

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("app_name", "", "App name")
dbutils.widgets.text("scope_name", "", "Secret scope")
dbutils.widgets.text("client_id_key", "", "M2M client ID key in scope")
dbutils.widgets.text("catalog", "", "UC catalog")
dbutils.widgets.text("schema_name", "", "UC schema (fully qualified)")
dbutils.widgets.text("mcp_service_name", "", "MCP service full name (catalog.schema.service)")
dbutils.widgets.text("visual_checks_volume", "", "ui_visual_checks volume full name (catalog.schema.volume); empty = skip")

app_name = dbutils.widgets.get("app_name")
scope_name = dbutils.widgets.get("scope_name")
client_id_key = dbutils.widgets.get("client_id_key")
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema_name")
mcp_service_name = dbutils.widgets.get("mcp_service_name")
visual_checks_volume = dbutils.widgets.get("visual_checks_volume").strip()

assert app_name, "app_name is required"
assert scope_name, "scope_name is required"
assert client_id_key, "client_id_key is required"
assert catalog, "catalog is required"
assert schema_name, "schema_name is required"
assert mcp_service_name, "mcp_service_name is required"

print(f"App:              {app_name}")
print(f"Scope:            {scope_name}")
print(f"Client ID key:    {client_id_key}")
print(f"Catalog:          {catalog}")
print(f"Schema:           {schema_name}")
print(f"MCP service:      {mcp_service_name}")
print(f"Visual checks vol: {visual_checks_volume or '(skipped)'}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Read M2M SPN client_id from secret scope
m2m_client_id = dbutils.secrets.get(scope=scope_name, key=client_id_key)
print(f"M2M SPN client_id: {m2m_client_id}")

# Look up SPN by applicationId
spn = None
for sp in w.service_principals.list(filter=f'applicationId eq "{m2m_client_id}"'):
    spn = sp
    break

if not spn:
    raise ValueError(f"Service principal not found for client_id={m2m_client_id}")

print(f"Found SPN: {spn.display_name} (id={spn.id}, appId={spn.application_id})")

In [0]:
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

result = w.apps.update_permissions(
    app_name=app_name,
    access_control_list=[
        AccessControlRequest(
            # service_principal_name expects the application_id (UUID), NOT the display name.
            # The Permissions API silently ignores unrecognized principal names.
            service_principal_name=spn.application_id,
            permission_level=PermissionLevel.CAN_USE,
        )
    ],
)
print(f"\u2713 Granted CAN_USE to {spn.display_name} (appId={spn.application_id}) on {app_name}")

In [0]:
from databricks.sdk.service.workspace import AclPermission

# Safety net: ensure the app's own SPN has READ on the secret scope.
# This mirrors the infra bundle's permissions block (session 18) and covers
# cases where an infra deploy runs without the app SPN variable.
app_info = w.apps.get(app_name)
app_spn_client_id = app_info.service_principal_client_id
print(f"App SPN client_id: {app_spn_client_id}")

w.secrets.put_acl(
    scope=scope_name,
    principal=app_spn_client_id,
    permission=AclPermission.READ,
)
print(f"\u2713 Granted READ on secret scope '{scope_name}' to app SPN {app_spn_client_id}")

# Print resulting ACLs
print(f"\nSecret scope '{scope_name}' ACL list:")
for acl in w.secrets.list_acls(scope=scope_name):
    print(f"  {acl.principal}: {acl.permission.value}")

In [0]:
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

# Additive: does not remove existing grants (M2M SPN keeps CAN_USE)
result = w.apps.update_permissions(
    app_name=app_name,
    access_control_list=[
        AccessControlRequest(
            group_name="users",
            permission_level=PermissionLevel.CAN_USE,
        )
    ],
)
print(f"\u2713 Granted CAN_USE to group 'users' on {app_name}")

In [0]:
from databricks.sdk.service import catalog as c

# account users does NOT include service principals.
# The M2M SPN needs explicit UC grants to invoke the MCP service:
#   USE CATALOG  → navigate to the schema
#   USE SCHEMA   → navigate to the service
#   EXECUTE      → invoke the MCP service
#   READ/WRITE VOLUME on ui_visual_checks → screenshots of the Chromium gate (optional)
uc_targets = [
    ("catalog", catalog, [c.Privilege.USE_CATALOG]),
    ("schema", schema_name, [c.Privilege.USE_SCHEMA]),
    ("mcp_service", mcp_service_name, [c.Privilege.EXECUTE]),
]
if visual_checks_volume:
    uc_targets.append(("volume", visual_checks_volume, [c.Privilege.READ_VOLUME, c.Privilege.WRITE_VOLUME]))

for securable_type, full_name, privileges in uc_targets:
    w.grants.update(
        securable_type=securable_type,
        full_name=full_name,
        changes=[
            c.PermissionsChange(
                # UC Grants API also requires application_id, not display_name
                principal=spn.application_id,
                add=privileges,
            )
        ],
    )
    print(f"\u2713 Granted {[p.value for p in privileges]} on {securable_type} {full_name} to {spn.display_name} ({spn.application_id})")

In [0]:
import json

# --- App permissions ---
print("App permissions:")
perms = w.apps.get_permissions(app_name)
grants = []
for acl in perms.access_control_list or []:
    principal = acl.user_name or acl.group_name or acl.service_principal_name or "unknown"
    levels = [p.permission_level.value for p in (acl.all_permissions or [])]
    grants.append({"principal": principal, "permissions": levels})
    print(f"  {principal}: {levels}")

# --- UC grants ---
print(f"\nUC grants for {spn.display_name}:")
uc_grants = {}
for securable_type, full_name, _ in uc_targets:
    g = w.grants.get(securable_type=securable_type, full_name=full_name)
    for pa in g.privilege_assignments or []:
        # UC reports SPN principals by application_id; match the display name too for safety
        if pa.principal in (spn.application_id, spn.display_name):
            privs = [p.value for p in pa.privileges]
            uc_grants[f"{securable_type}:{full_name}"] = privs
            print(f"  {securable_type} {full_name}: {privs}")

dbutils.notebook.exit(json.dumps({"app_name": app_name, "grants": grants, "uc_grants": uc_grants}))